In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [5]:
data=read.csv('Daily_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [6]:
dim(data)

[1] 4227   23

In [7]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9977307,2.180566e-11,31.265883,4.202931,0.8853252,3.0007563,0.009502084,⋯,0.007626758,-0.5188252,0.2734535,1006,0.02104115,0.02104115,0.02104115,0,0,0
2,1,0,1,0.9753638,2.434650e-09,24.625465,-1.039634,0.8973350,3.0475729,0.211225819,⋯,0.004072618,-0.5203490,0.2801685,1006,0.02027750,0.02027750,0.02027750,0,0,0
3,1,0,1,0.8937784,8.333451e-06,3.804175,4.843726,0.4029370,0.3356327,0.610410200,⋯,0.002531199,-0.4987115,0.2487157,130,0.01750016,0.01750016,0.01750016,0,0,0
4,1,0,1,0.7424974,8.486237e-06,1.762132,6.219671,0.4733951,0.4720488,0.740639635,⋯,0.058828311,-0.5022730,0.3038757,169,0.02131033,0.02131033,0.02131033,0,0,0
5,1,0,1,0.9764692,1.782445e-07,11.344076,0.859238,0.6058021,0.8285215,0.281090710,⋯,0.122022465,-0.4771979,0.3813929,156,0.15410614,0.15410614,0.15410614,0,0,0
6,1,0,1,0.9963228,8.446153e-11,29.701579,10.112177,0.8651846,3.0647427,0.043039068,⋯,0.076311068,-0.5852963,0.4167008,1006,0.01933837,0.01933837,0.01933837,0,0,0


In [9]:
dlist= load('Daily_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [10]:
dim(MASE)

[1] 4227    5    4

In [11]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [12]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [13]:
nanum

NULL

In [14]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [15]:
table(realbestmin)

realbestmin
   0    1    2    3    4 
 637  588  520  604 1878 

In [16]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [17]:
realbestmean

4
2
1
3
0
4
4
2
2
4
4


In [18]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9977307,2.180566e-11,31.265883,4.202931,0.8853252,3.0007563,0.009502084,⋯,0.007626758,-0.5188252,0.2734535,1006,0.02104115,0.02104115,0.02104115,0,0,0
2,1,0,1,0.9753638,2.434650e-09,24.625465,-1.039634,0.8973350,3.0475729,0.211225819,⋯,0.004072618,-0.5203490,0.2801685,1006,0.02027750,0.02027750,0.02027750,0,0,0
3,1,0,1,0.8937784,8.333451e-06,3.804175,4.843726,0.4029370,0.3356327,0.610410200,⋯,0.002531199,-0.4987115,0.2487157,130,0.01750016,0.01750016,0.01750016,0,0,0
4,1,0,1,0.7424974,8.486237e-06,1.762132,6.219671,0.4733951,0.4720488,0.740639635,⋯,0.058828311,-0.5022730,0.3038757,169,0.02131033,0.02131033,0.02131033,0,0,0
5,1,0,1,0.9764692,1.782445e-07,11.344076,0.859238,0.6058021,0.8285215,0.281090710,⋯,0.122022465,-0.4771979,0.3813929,156,0.15410614,0.15410614,0.15410614,0,0,0
6,1,0,1,0.9963228,8.446153e-11,29.701579,10.112177,0.8651846,3.0647427,0.043039068,⋯,0.076311068,-0.5852963,0.4167008,1006,0.01933837,0.01933837,0.01933837,0,0,0


In [19]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [20]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [21]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9977307,2.180566e-11,31.265883,4.202931,0.8853252,3.0007563,0.009502084,⋯,0.007626758,-0.5188252,0.2734535,1006,0.02104115,0.02104115,0.02104115,0,0,0
2,1,0,1,0.9753638,2.434650e-09,24.625465,-1.039634,0.8973350,3.0475729,0.211225819,⋯,0.004072618,-0.5203490,0.2801685,1006,0.02027750,0.02027750,0.02027750,0,0,0
3,1,0,1,0.8937784,8.333451e-06,3.804175,4.843726,0.4029370,0.3356327,0.610410200,⋯,0.002531199,-0.4987115,0.2487157,130,0.01750016,0.01750016,0.01750016,0,0,0
4,1,0,1,0.7424974,8.486237e-06,1.762132,6.219671,0.4733951,0.4720488,0.740639635,⋯,0.058828311,-0.5022730,0.3038757,169,0.02131033,0.02131033,0.02131033,0,0,0
5,1,0,1,0.9764692,1.782445e-07,11.344076,0.859238,0.6058021,0.8285215,0.281090710,⋯,0.122022465,-0.4771979,0.3813929,156,0.15410614,0.15410614,0.15410614,0,0,0
6,1,0,1,0.9963228,8.446153e-11,29.701579,10.112177,0.8651846,3.0647427,0.043039068,⋯,0.076311068,-0.5852963,0.4167008,1006,0.01933837,0.01933837,0.01933837,0,0,0


In [22]:
end_time = Sys.time()

In [23]:
time_matrix[1,]=end_time-start_time

In [24]:
end_time-start_time

Time difference of 3.256759 mins

## Target the interval where the actual error is minimum

In [25]:
start_time = Sys.time()

In [26]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [27]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [28]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:2.027987 
[2]	train-rmse:1.662826 
[3]	train-rmse:1.440380 
[4]	train-rmse:1.303442 
[5]	train-rmse:1.210462 
[6]	train-rmse:1.164064 
[7]	train-rmse:1.119478 
[8]	train-rmse:1.090350 
[9]	train-rmse:1.064084 
[10]	train-rmse:1.049398 
[11]	train-rmse:1.026206 
[12]	train-rmse:1.012451 
[13]	train-rmse:1.006077 
[14]	train-rmse:0.996633 
[15]	train-rmse:0.986988 
[16]	train-rmse:0.973306 
[17]	train-rmse:0.963490 
[18]	train-rmse:0.957566 
[19]	train-rmse:0.947984 
[20]	train-rmse:0.929107 
[21]	train-rmse:0.913041 
[22]	train-rmse:0.907202 
[23]	train-rmse:0.891014 
[24]	train-rmse:0.888458 
[25]	train-rmse:0.865180 
[26]	train-rmse:0.842907 
[27]	train-rmse:0.826423 
[28]	train-rmse:0.810687 
[29]	train-rmse:0.797543 
[30]	train-rmse:0.793984 
[31]	train-rmse:0.790695 
[32]	train-rmse:0.779022 
[33]	train-rmse:0.762602 
[34]	train-rmse:0.747908 
[35]	train-rmse:0.735655 
[36]	train-rmse:0.726606 
[37]	train-rmse:0.713057 
[38]	train-rmse:0.709224 
[39]	train-rmse:0.699

In [29]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.410376 
[2]	train-mlogloss:1.282364 
[3]	train-mlogloss:1.189361 
[4]	train-mlogloss:1.118990 
[5]	train-mlogloss:1.062103 
[6]	train-mlogloss:1.018212 
[7]	train-mlogloss:0.982171 
[8]	train-mlogloss:0.943813 
[9]	train-mlogloss:0.912143 
[10]	train-mlogloss:0.884694 
[11]	train-mlogloss:0.864693 
[12]	train-mlogloss:0.839466 
[13]	train-mlogloss:0.823940 
[14]	train-mlogloss:0.807229 
[15]	train-mlogloss:0.790135 
[16]	train-mlogloss:0.775282 
[17]	train-mlogloss:0.752963 
[18]	train-mlogloss:0.733967 
[19]	train-mlogloss:0.713186 
[20]	train-mlogloss:0.698253 
[21]	train-mlogloss:0.681386 
[22]	train-mlogloss:0.666680 
[23]	train-mlogloss:0.652076 
[24]	train-mlogloss:0.641488 
[25]	train-mlogloss:0.631870 
[26]	train-mlogloss:0.620209 
[27]	train-mlogloss:0.606954 
[28]	train-mlogloss:0.597773 
[29]	train-mlogloss:0.588215 
[30]	train-mlogloss:0.576998 
[31]	train-mlogloss:0.565451 
[32]	train-mlogloss:0.558973 
[33]	train-mlogloss:0.548585 
[34]	train-mlogloss

In [30]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007420 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4321
[LightGBM] [Info] Number of data points in the train set: 2968, number of used features: 17
[LightGBM] [Info] Start training from score 2.599057


In [31]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002479 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4321
[LightGBM] [Info] Number of data points in the train set: 2968, number of used features: 17
[LightGBM] [Info] Start training from score -1.890850
[LightGBM] [Info] Start training from score -1.991757
[LightGBM] [Info] Start training from score -2.090282
[LightGBM] [Info] Start training from score -1.962557
[LightGBM] [Info] Start training from score -0.801958


In [32]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [33]:
start_time = Sys.time()

In [34]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [35]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [36]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.869672 
[2]	train-rmse:1.571892 
[3]	train-rmse:1.379696 
[4]	train-rmse:1.268615 
[5]	train-rmse:1.192463 
[6]	train-rmse:1.142021 
[7]	train-rmse:1.100392 
[8]	train-rmse:1.071533 
[9]	train-rmse:1.052859 
[10]	train-rmse:1.041077 
[11]	train-rmse:1.017936 
[12]	train-rmse:1.012150 
[13]	train-rmse:1.007281 
[14]	train-rmse:0.985220 
[15]	train-rmse:0.976359 
[16]	train-rmse:0.969913 
[17]	train-rmse:0.958445 
[18]	train-rmse:0.943752 
[19]	train-rmse:0.930829 
[20]	train-rmse:0.922388 
[21]	train-rmse:0.912746 
[22]	train-rmse:0.898744 
[23]	train-rmse:0.885598 
[24]	train-rmse:0.869420 
[25]	train-rmse:0.860246 
[26]	train-rmse:0.845617 
[27]	train-rmse:0.835644 
[28]	train-rmse:0.831253 
[29]	train-rmse:0.821113 
[30]	train-rmse:0.807212 
[31]	train-rmse:0.801160 
[32]	train-rmse:0.792933 
[33]	train-rmse:0.779930 
[34]	train-rmse:0.769462 
[35]	train-rmse:0.760781 
[36]	train-rmse:0.743212 
[37]	train-rmse:0.728918 
[38]	train-rmse:0.714716 
[39]	train-rmse:0.708

In [37]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.443240 
[2]	train-mlogloss:1.331990 
[3]	train-mlogloss:1.246444 
[4]	train-mlogloss:1.179185 
[5]	train-mlogloss:1.117808 
[6]	train-mlogloss:1.064281 
[7]	train-mlogloss:1.016289 
[8]	train-mlogloss:0.979897 
[9]	train-mlogloss:0.942395 
[10]	train-mlogloss:0.911975 
[11]	train-mlogloss:0.888183 
[12]	train-mlogloss:0.865912 
[13]	train-mlogloss:0.848875 
[14]	train-mlogloss:0.831375 
[15]	train-mlogloss:0.813440 
[16]	train-mlogloss:0.799178 
[17]	train-mlogloss:0.781657 
[18]	train-mlogloss:0.768463 
[19]	train-mlogloss:0.749398 
[20]	train-mlogloss:0.729440 
[21]	train-mlogloss:0.718572 
[22]	train-mlogloss:0.706553 
[23]	train-mlogloss:0.688277 
[24]	train-mlogloss:0.666778 
[25]	train-mlogloss:0.649513 
[26]	train-mlogloss:0.637823 
[27]	train-mlogloss:0.621786 
[28]	train-mlogloss:0.611763 
[29]	train-mlogloss:0.600821 
[30]	train-mlogloss:0.589723 
[31]	train-mlogloss:0.575543 
[32]	train-mlogloss:0.563179 
[33]	train-mlogloss:0.549655 
[34]	train-mlogloss

In [38]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007234 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4321
[LightGBM] [Info] Number of data points in the train set: 2968, number of used features: 17
[LightGBM] [Info] Start training from score 2.246294


In [39]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002467 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4321
[LightGBM] [Info] Number of data points in the train set: 2968, number of used features: 17
[LightGBM] [Info] Start training from score -1.580547
[LightGBM] [Info] Start training from score -1.739894
[LightGBM] [Info] Start training from score -2.149205
[LightGBM] [Info] Start training from score -1.769107
[LightGBM] [Info] Start training from score -1.104018


In [40]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [41]:
end_time-start_time

Time difference of 5.932424 secs

## predict

In [42]:
start_time = Sys.time()

In [43]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [44]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [45]:
xgbregmin

[1]  3.8050947189  2.9539706707  2.9009854794  2.1128444672  0.2093249559
   [6]  3.9554514885  3.1195542812  2.2763259411  2.1558077335  3.9081592560
  [11]  3.4743554592  1.9174026251  1.5356433392  3.8615937233  4.4298548698
  [16]  0.2475445867  0.6854084730  1.0544877052  3.9170591831  0.0215129126
  [21]  4.1619005203  1.5611861944  1.7255460024  1.5988490582  3.7279462814
  [26]  2.0625855923  2.4640426636  3.3732721806  4.1897306442  3.4421882629
  [31]  0.1159796268 -1.0288259983  3.8937888145  3.9593715668  3.0379383564
  [36]  4.6947488785  1.9985797405  3.7179219723  2.5038027763  0.2245841622
  [41]  3.9702107906  1.1118648052  2.4955604076  2.1406354904  0.5684389472
  [46]  3.0439021587  2.7377247810  4.6553254128  3.0625772476  3.9265530109
  [51]  3.8834211826  3.3439126015  0.7295902371  3.6342449188  0.3841051459
  [56]  3.1216142178  1.2646620274  2.8540012836  3.9212980270  2.8045797348
  [61]  1.4485447407  1.3101063967  3.3308172226  1.3142261505  2.5541739464
  [66]  3.6738271713  3.6224870682  2.1039509773  0.3449681401  4.0124206543
  [71]  2.9925587177  0.3487843275  3.8631832600  1.4761997461  2.8502659798
  [76]  3.8317558765  3.5476760864  3.6950354576  4.3641362190  2.2174148560
  [81]  3.8958210945  3.9885537624  3.5567314625  2.2007606030  3.7652387619
  [86]  3.7676787376  3.3970570564  3.9624183178  3.8531291485  3.2547268867
  [91]  2.8723175526  3.8246636391  3.9614138603  1.1135541201  2.4687488079
  [96]  3.7301933765  1.3650264740  3.7758142948  2.1796917915  3.4695072174
 [101]  3.8891482353  3.8740110397  3.6606204510  3.4400212765  0.5996722579
 [106]  1.8403687477  1.2205624580  2.8447530270  3.6661772728  3.8049731255
 [111]  3.3483190536  2.6692159176  3.7414577007  3.0220866203  2.0405004025
 [116]  3.0127506256  3.4121363163  1.4738326073  0.6312718987  2.8453738689
 [121]  3.5015327930  3.6724493504  2.5452516079  1.1787153482  0.2468389124
 [126]  0.5595763922  2.7446455956  3.3645699024  3.7888879776  1.3244317770
 [131]  3.0863845348  3.5686929226  3.6026096344  3.0211677551  2.9288091660
 [136]  3.8051166534  2.9043691158  4.0261740685  0.3617223501  2.0257363319
 [141]  2.8347733021  1.9705276489  1.3169766665  3.8052310944  3.6487469673
 [146]  0.2493648529  0.5206634998  2.3288404942  2.5179605484  3.0368216038
 [151]  2.7594015598  1.5656962395  3.4310591221  3.1423976421  2.3420557976
 [156]  1.7267899513  0.6067681313  0.8525158167  3.7633035183  1.9913452864
 [161]  1.0184620619  3.7990248203  1.5077055693  1.0388263464  1.2895313501
 [166]  4.0696377754  0.0756204054  1.7718856335  2.9550967216  1.9721970558
 [171]  0.4940018952  0.6073154211  0.2472689450  1.9521934986  2.9563982487
 [176]  3.7522664070  2.5890750885  4.0912275314  0.8008166552  2.0215709209
 [181]  2.2788116932  1.9473551512  1.8530443907  3.8812735081  3.9394969940
 [186]  3.8273227215  3.0452990532  3.8060066700  2.2649259567  1.0336900949
 [191]  1.9088591337  3.0776469707  2.2710504532  4.1765398979  3.6659789085
 [196]  4.6136121750  0.7797375321  0.4025285840  0.4419832230  3.0455424786
 [201]  3.8887674809  2.5988323689  3.1400480270  2.8472774029  1.0233656168
 [206]  3.6924803257  3.0388233662  3.9882607460  3.8451313972  4.0652499199
 [211]  3.9095888138  1.8094965219  3.4011166096  0.3371165395  3.2381844521
 [216]  3.4928781986  0.7172096968  3.1736104488  3.5805099010  1.3568288088
 [221]  0.0797620639  1.3935524225  3.8706982136  3.0526862144  3.9524605274
 [226]  0.4766156971  0.0912410915  0.4606320262  2.7342255116  3.9180822372
 [231]  4.1240935326  3.5269458294  1.8268767595  2.4070074558  2.9845068455
 [236]  3.1152892113  2.0242180824  3.9477031231  4.2341432571  4.0717921257
 [241]  0.7923899889  3.4991703033  3.1859626770  3.3330771923  2.9779980183
 [246]  3.9550554752  3.1427695751  1.7876358032  0.6035523415  0.1583186835
 [251] -0.0110440925  1.9568163157  1.8357012272  1.2738997936  2.3055155277
 [256]  4.4429492950  1.2092528343  1.5461955070  1.4677888155  1.8556555510

In [46]:
xgbclsmin

[1] 4 3 3 2 0 4 4 2 2 4 4 4 1 4 4 0 0 0 4 0 4 2 1 2 4 2 4 4 4 4 0 0 4 4 3 4 2
  [38] 4 4 0 4 4 4 4 0 3 3 3 3 4 4 4 0 4 0 3 0 3 4 3 1 1 4 1 3 4 4 2 0 4 3 0 4 4
  [75] 3 4 4 4 4 2 4 4 4 4 4 4 4 4 4 4 4 4 4 1 1 4 1 4 2 4 4 4 4 4 0 2 2 4 4 4 4
 [112] 4 4 3 4 3 4 0 0 4 4 4 3 1 0 0 4 4 4 1 3 4 2 3 3 4 3 4 0 2 3 2 1 4 4 1 2 2
 [149] 3 3 2 1 4 3 2 4 4 0 4 2 4 4 1 1 1 4 0 4 4 0 0 0 0 4 0 4 4 4 0 2 2 4 4 4 4
 [186] 4 3 4 2 2 4 3 1 4 4 4 0 0 0 4 4 4 4 4 1 2 4 4 4 4 4 4 4 0 4 4 2 4 4 1 0 1
 [223] 4 4 4 0 0 0 3 4 4 4 4 3 3 3 2 4 4 4 0 3 3 3 3 4 3 1 0 0 0 1 1 1 1 4 1 1 1
 [260] 2 4 4 2 4 4 4 4 4 4 4 0 4 0 4 3 4 2 4 1 4 4 4 4 0 3 4 4 4 4 4 4 4 4 2 0 0
 [297] 0 0 4 4 4 0 4 3 3 4 4 3 1 4 1 1 2 4 4 4 2 0 0 0 0 0 0 0 0 2 2 2 2 2 1 1 1
 [334] 1 1 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 3 0 3 1 0 4 0 4 4 3 4 4 0 0 3 3 4 4 4
 [371] 4 1 3 1 2 4 4 4 4 4 4 4 4 4 2 2 4 4 0 2 1 4 4 2 2 4 4 4 4 3 2 3 4 1 2 0 2
 [408] 4 2 3 2 4 0 1 0 4 3 4 0 4 4 4 4 4 4 4 0 3 0 2 4 0 1 3 0 4 4 0 2 2 4 2 1 4
 [445] 4 0 4 3 4 2 1 1 4 4 4 4 4 4 4 4 1 4 4 4 4 1 3 2 2 4 2 3 3 4 1 4 1 4 4 4 4
 [482] 2 1 3 4 2 1 4 4 4 2 4 2 4 3 4 4 4 2 4 1 2 1 0 0 4 0 1 1 3 4 1 0 0 0 0 4 0
 [519] 1 0 0 1 3 3 3 2 2 4 2 3 4 2 2 2 4 4 3 3 2 4 4 3 2 1 0 2 1 1 1 4 1 2 1 2 2
 [556] 4 0 0 4 0 4 4 0 4 4 4 4 4 4 4 0 2 2 4 1 3 4 3 4 2 2 0 4 2 4 4 4 4 0 0 4 4
 [593] 4 4 4 0 4 3 4 4 4 4 4 0 4 4 4 4 4 4 4 0 4 0 0 0 0 0 0 0 0 0 0 0 0 4 0 2 1
 [630] 3 4 4 4 2 3 3 3 4 4 4 2 2 0 4 3 2 4 4 1 0 2 3 3 2 2 3 2 2 3 4 4 2 2 2 4 3
 [667] 3 1 4 2 2 0 4 2 4 1 4 1 4 3 4 4 0 4 4 3 4 3 4 2 0 4 4 4 4 4 2 2 4 4 1 4 4
 [704] 4 1 4 4 1 4 4 4 4 1 4 3 1 4 4 4 4 4 4 4 4 1 4 2 4 4 4 4 4 3 3 4 4 4 4 4 1
 [741] 0 0 4 0 0 0 2 4 4 0 0 2 0 0 0 2 0 1 2 1 2 4 1 2 4 0 1 2 3 4 0 1 0 3 3 4 3
 [778] 1 4 1 4 2 0 3 1 0 3 1 0 4 2 4 1 4 1 3 4 0 4 4 4 4 4 4 4 4 4 4 3 4 4 0 4 4
 [815] 4 4 4 1 4 4 4 4 4 2 4 4 4 1 1 2 4 2 4 2 4 3 4 0 4 4 3 4 0 4 2 4 4 3 2 2 4
 [852] 0 1 3 3 4 4 4 0 4 4 4 4 4 4 2 1 1 1 0 0 4 3 2 1 1 2 0 1 0 2 1 0 0 2 0 1 1
 [889] 1 4 1 1 4 0 0 4 1 2 1 4 1 4 4 4 4 1 1 4 1 1 1 1 4 4 1 3 4 3 3 2 4 4 3 4 4
 [926] 4 4 4 4 4 3 4 4 4 4 4 4 4 4 4 4 0 4 1 1 4 4 1 3 1 2 0 3 2 2 0 2 1 0 3 0 0
 [963] 2 1 0 0 1 2 1 2 1 0 4 4 1 2 1 1 1 4 2 0 1 2 1 2 3 3 1 4 0 2 1 4 1 4 0 0 1
[1000] 4 0 4 1 2 0 0 4 4 3 4 3 4 1 4 4 2 1 3 4 4 0 4 4 4 0 4 0 0 4 2 0 0 4 0 3 0
[1037] 3 0 0 1 1 0 4 0 4 4 0 4 4 4 3 3 4 1 3 2 1 3 4 4 0 4 4 4 4 3 4 2 4 4 4 0 4
[1074] 0 1 4 4 0 0 0 4 0 1 1 4 0 0 0 1 0 1 0 1 1 4 4 2 4 4 4 3 4 4 4 4 4 2 4 4 4
[1111] 4 1 0 4 1 4 4 4 4 4 4 2 2 3 4 4 4 1 2 2 1 1 2 1 4 2 1 2 1 1 0 4 3 0 1 1 2
[1148] 4 4 1 2 2 0 1 1 0 0 0 0 0 0 2 0 1 0 1 2 0 0 0 1 0 1 0 2 1 4 0 2 0 1 2 2 0
[1185] 1 1 4 4 2 4 4 4 4 0 0 4 4 2 4 0 4 4 4 4 4 4 4 4 4 4 4 4 4 4 0 4 4 4 4 3 4
[1222] 4 4 0 4 1 4 4 4 4 2 2 0 4 1 1 2 2 1 2 4 1 0 0 4 2 2 4 3 2 2 1 4 2 0 4 1 1
[1259] 1 0 1 1 1 0 1 1 2 0 2 1 2 2 1 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
[1296] 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 0 0 4 2 0 0 2 0 2 4 2 4 0 2 0 0 4 1 4 2 2
[1333] 4 2 2 0 2 4 0 4 4 4 4 4 4 1 1 1 1 0 4 0 2 4 2 4 4 0 4 1 4 1 0 2 2 0 2 2 2
[1370] 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 2 1 2 1 0 1 1 1 1 0 0 0 0 0 0 0 1 1 2 1 0 0
[1407] 0 1 0 2 0 0 0 0 0 0 0 4 0 0 1 1 0 1 0 2 1 4 4 4 4 4 4 0 4 2 3 4 4 4 4 4 4
[1444] 0 1 4 4 3 4 2 1 4 3 4 2 4 4 3 3 4 4 2 2 3 0 2 0 4 4 4 4 4 4 4 4 4 3 4 4 4
[1481] 4 4 4 2 4 4 2 4 4 4 4 4 4 4 0 4 3 0 4 0 4 4 1 3 0 3 4 4 4 0 4 0 4 0 4 4 0
[1518] 1 4 4 4 4 4 0 0 0 4 4 2 1 1 2 2 2 1 1 1 1 1 2 2 2 2 1 2 0 1 1 2 2 2 1 1 2
[1555] 2 2 2 1 1 2 2 2 2 2 2 1 2 2 1 2 2 1 4 4 2 2 4 1 2 4 2 2 1 1 1 1 1 2 4 2 1
[1592] 2 1 2 1 4 2 1 4 2 2 4 4 4 4 0 4 4 4 3 3 4 4 1 4 0 1 4 1 3 3 4 4 1 4 4 4 4
[1629] 4 2 1 4 4 0 4 4 4 4 4 4 3 3 4 4 4 2 2 2 4 4 3 4 4 4 0 0 2 4 4 4 0 0 4 4 4
[1666] 0 4 4 0 4 4 4 4 1 4 4 4 4 4 2 4 2 4 3 4 4 4 3 4 4 4 4 4 4 4 4 4 0 3 2 4 4
[1703] 4 4 2 1 4 3 4 4 4 4 4 2 4 4 4 4 3 1 4 4 4 0 4 2 0 4 1 4 4 4 4 4 4 4 4 4 4
[1740] 3 4 4 4 4 4 3 4 3 3 2 4 3 0 4 1 4 2 4 4 4 3 4 3 1 4 0 4 3 4 4 4 4 2 0 4 4
[1777] 0 4 4 4 4 4 4 1 3 4 0 3 4 4 4 0 2 0 2 2 4 3 3 0 3 4 4 4 0 4 4 4 4 4 4 3 4
[1814] 2 4 4 4 4 0 4 4 4 4 4 4 0 2

In [47]:
lgbregmin

[1]  3.908699752  2.811167017  2.122408565  2.024357700  1.601400500
   [6]  3.700863896  2.905236313  2.880446707  2.770436432  3.545700375
  [11]  3.006386714  3.115538703  2.120199549  3.945020149  3.987741193
  [16]  0.677609329  1.794359417  2.188628918  3.523816210  1.289312585
  [21]  3.797052867  3.035090822  2.564922203  3.048062839  2.976481621
  [26]  2.676860682  3.207340059  2.647774593  3.893491612  2.938992299
  [31]  1.148423603  1.583604586  4.018017169  3.589322998  2.918089809
  [36]  2.945494383  1.634551519  3.431066983  2.708417531  1.350464309
  [41]  3.309502179  1.602327799  2.686735332  2.846390969  1.352154360
  [46]  3.047518281  2.472934620  3.713092200  3.145919221  3.716552131
  [51]  3.573815875  3.058185199  1.334817290  3.024913816  1.798231724
  [56]  3.666221879  1.464074036  2.484322794  3.602349075  3.221910281
  [61]  1.921225531  1.762840882  3.772681996  1.885936048  2.371884666
  [66]  3.087071678  3.448749451  2.373900922  0.741275697  3.790724279
  [71]  2.779586781  1.627476343  3.318730704  1.568944599  2.582542413
  [76]  3.439832080  3.399038597  3.067117970  3.255479116  2.950952895
  [81]  3.178713151  3.150797633  2.922463721  2.135030206  3.089517643
  [86]  3.883222007  3.810458217  3.895811566  3.826201497  3.665676416
  [91]  3.833637949  3.224354214  3.845404337  1.610088485  1.897890001
  [96]  3.741599322  1.694133967  3.584517698  1.913694771  3.205596708
 [101]  3.739456183  3.602893419  3.028812661  3.158417716  2.172125546
 [106]  1.486200039  2.176917828  4.312655090  3.500292123  3.515872913
 [111]  4.102433684  4.185548750  3.404939818  2.738614440  3.303459944
 [116]  2.775230995  3.187038161  1.260991909  0.794706256  3.904929981
 [121]  3.165967298  3.532333994  2.563071648  1.256758788  0.744773721
 [126]  1.240936259  3.641941552  3.124911997  2.748197315  1.548033859
 [131]  2.631196577  3.499895269  3.492991155  3.107529308  2.874502726
 [136]  3.104192016  2.764897802  3.495443348  1.162425816  2.232581667
 [141]  2.216449479  2.236783695  1.899950412  3.623774243  3.130322395
 [146]  1.556386177  1.670294730  2.747546784  2.197631809  2.913331369
 [151]  2.715220865  2.417209281  2.918970396  3.321293449  2.411888182
 [156]  3.074383789  3.738062932  0.796373912  2.985115097  1.843908665
 [161]  3.355211510  2.817734501  2.052827023  1.226368132  1.264761145
 [166]  3.631658793  1.795700698  2.169494250  3.586715928  2.487418904
 [171]  0.982602458  0.978229365  0.926040436  2.254886994  1.905051402
 [176]  3.717711074  4.330422004  3.712350451  1.167940836  2.460979407
 [181]  2.071956200  2.914434740  4.224000244  3.775969258  3.928606789
 [186]  3.595175202  3.247422209  3.706676942  3.324844780  3.288355917
 [191]  3.784710433  3.158827072  2.187517683  4.509148689  3.917677473
 [196]  3.812916831  1.531497915  1.571781798  1.469920891  2.922921338
 [201]  3.359568169  2.404219401  3.113322204  2.668823882  2.155066073
 [206]  2.942960746  2.962904073  3.762171584  3.752957606  3.950189310
 [211]  4.233667641  2.660925281  2.713904607  1.609961490  2.531657423
 [216]  3.424795897  2.176845487  2.544295756  2.839426588  2.221467243
 [221]  0.561460938  1.986195533  3.605240690  3.817552937  3.813728215
 [226]  0.771894738  0.945139694  1.096560250  2.101046327  4.138724537
 [231]  3.828503119  3.868362658  1.953307548  2.587935394  3.128767024
 [236]  3.163136846  2.487850317  3.673900387  3.432873829  3.140610878
 [241]  1.891905470  3.416823981  3.172500155  3.386449890  2.952812251
 [246]  3.166508662  3.228634928  2.207750768  0.723477236  0.265608427
 [251]  0.743588394  1.738197060  1.922173853  1.846837530  2.288076064
 [256]  3.441960108  1.894892986  1.690314945  1.802952357  2.807503209
 [261]  3.102593817  2.755939404  2.388413931  2.830338809  2.707835432
 [266]  2.602378835  3.113358950  2.913668207  2.581651376  1.853041567
 [271]  1.045778724  2.343324314  1.218981408  3.040751558  2.545984811
 [276]  2.986169282  2.600928362  3.3658850

In [48]:
lgbclsmin

0.006627230,0.0033924692,0.004297776,0.044227054,0.94145547
0.007316336,0.0205559440,0.022692825,0.882527209,0.06690769
0.121833622,0.0057535702,0.019190618,0.815186676,0.03803551
0.114244711,0.0205339520,0.763919122,0.021704876,0.07959734
0.794027808,0.0134338569,0.008624192,0.042255555,0.14165859
0.012722062,0.0018953211,0.023398506,0.007026098,0.95495801
0.150350846,0.0109875566,0.305192680,0.042764013,0.49070490
0.005771339,0.0178151475,0.687976428,0.061743222,0.22669386
0.015437138,0.0171101984,0.660491985,0.070562355,0.23639832
0.009692029,0.0175414250,0.040173020,0.070825601,0.86176792
0.027447858,0.0493557055,0.046206395,0.138625001,0.73836504


In [49]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [50]:
lgbclsminr

4
1
0
0
1
4
0
4
4
1
2


In [51]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [52]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [53]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [54]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
4,3.805095,4,3.908700
3,2.953971,1,2.811167
3,2.900985,0,2.122409
2,2.112844,0,2.024358
0,0.209325,1,1.601400
4,3.955451,4,3.700864


In [55]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [56]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
4,3.6884120,4,3.7138140
2,2.3273439,1,2.5378621
1,1.0572011,4,0.8504283
3,2.7177610,1,2.2087852
0,0.1347971,1,1.2520085
4,4.1197233,2,3.5999014


In [57]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
4,3.6884120,4,3.7138140
2,2.3273439,1,2.5378621
1,1.0572011,4,0.8504283
3,2.7177610,1,2.2087852
0,0.1347971,1,1.2520085
4,4.1197233,2,3.5999014


In [58]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [59]:
result_list=list(preallmin,preallmean,time_matrix)

In [63]:
save(result_list, file = "Daily_nnetar_opt_pre_result.RData")

In [62]:
time_matrix

user_time,system_time,elapsed_time
3.256759,3.256759,3.256759
7.354278,7.354278,7.354278
5.932424,5.932424,5.932424
2.570808,2.570808,2.570808


In [31]:
importance_matrix1 <- xgb.importance(model = xgb_mean_cl)
importance_matrix2 <- xgb.importance(model = xgb_mean_reg)
importance_matrix3 <- lgb.importance(lgb_mean_cl)
importance_matrix4 <- lgb.importance(lgb_mean_reg)

importance_matrix1=importance_matrix1[order(importance_matrix1$Feature),]
importance_matrix2=importance_matrix2[order(importance_matrix2$Feature),]
importance_matrix3=importance_matrix3[order(importance_matrix3$Feature),]
importance_matrix4=importance_matrix4[order(importance_matrix4$Feature),]

importance_matrix_mean=(importance_matrix1[,2:4]+importance_matrix2[,2:4]+importance_matrix3[,2:4]+importance_matrix4[,2:4])/4

importance=cbind(importance_matrix1[,1],importance_matrix_mean)

write.csv(importance,'d_ets_imp_mean.csv')